# 03 — Pipeline Figures

Visual documentation of the data collection and transformation pipeline.

**Manuscript section:** Methods

**Outputs:** `notebooks/data_paper/figures/fig_pipeline_flow.png`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from config.paths import REPO_ROOT

FIGURES_DIR = REPO_ROOT / "projects/litigancia/notebooks/data_paper/figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Pipeline flowchart

In [ ]:
def draw_box(ax, xy, text, width=2.8, height=0.9, fc="#eef4ff"):
    x, y = xy
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.05,rounding_size=0.08",
        linewidth=1.2,
        edgecolor="#355070",
        facecolor=fc,
    )
    ax.add_patch(box)
    ax.text(x + width / 2, y + height / 2, text, ha="center", va="center", fontsize=9, wrap=True)


def draw_arrow(ax, start, end):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="->",
            mutation_scale=12,
            linewidth=1.2,
            color="#355070",
        )
    )


fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 10)
ax.axis("off")
ax.set_title("TJSP Fiscal Execution Data Pipeline", fontsize=14, pad=16)

# Main processos track
draw_box(ax, (1.0, 8.0), "juscraper recoleta\nCOLLECT_ROOT\n(JSON + SQLite)")
draw_box(ax, (1.0, 6.3), "Bronze\ncoletas_delta\n(hash + path)")
draw_box(ax, (1.0, 4.6), "Silver staging\nDuckDB dedupe\n(source_bronze_path)")
draw_box(ax, (1.0, 2.9), "Silver\nprocessos_delta\n(~6.1M decisions)")

draw_arrow(ax, (2.4, 8.0), (2.4, 7.2))
draw_arrow(ax, (2.4, 6.3), (2.4, 5.5))
draw_arrow(ax, (2.4, 4.6), (2.4, 3.8))

# FACE track
draw_box(ax, (6.5, 8.0), "FACE scraper\nface_tjsp.py", fc="#f7f1ea")
draw_box(ax, (6.5, 6.3), "Bronze\nface_processos_delta", fc="#f7f1ea")
draw_box(ax, (6.5, 4.6), "Silver\nface_processos_clean_delta\n(~1.5M records)", fc="#f7f1ea")

draw_arrow(ax, (7.9, 8.0), (7.9, 7.2))
draw_arrow(ax, (7.9, 6.3), (7.9, 5.5))

# Movimentações
draw_box(ax, (6.5, 2.9), "Silver\nmovimentacoes_delta", fc="#f7f1ea")
draw_arrow(ax, (7.9, 4.6), (7.9, 3.8))

# Join annotation
draw_box(ax, (4.0, 1.2), "Join key: cd_processo", width=4.0, height=0.7, fc="#ffffff")
draw_arrow(ax, (2.4, 2.9), (4.8, 1.9))
draw_arrow(ax, (7.9, 4.6), (6.0, 1.9))

fig.tight_layout()
out = FIGURES_DIR / "fig_pipeline_flow.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved {out}")

## Mermaid source (for manuscript / docs)

Copy into manuscript or GitHub-flavoured markdown:

```mermaid
flowchart TB
  collect["juscraper recoleta\nCOLLECT_ROOT JSON/SQLite"]
  bronzeP["bronze coletas_delta"]
  silverP["silver processos_delta"]
  faceScrape["FACE scraper"]
  bronzeF["bronze face_processos_delta"]
  silverF["silver face_processos_clean_delta"]
  silverM["silver movimentacoes_delta"]

  collect --> bronzeP --> silverP
  faceScrape --> bronzeF --> silverF --> silverM
  silverP --- silverF
```